# 构建rag应用

## 1调用向量数据库
传入文档构建数据库时使用Chroma.from_documents，
对已经建好的数据库调用直接Chroma
 

In [2]:
#加载密匙到环境变量
import os
from dotenv import find_dotenv,load_dotenv
_ = load_dotenv(load_dotenv())
api_key = os.environ["ZHIPUAI_API_KEY"]

#定义Embedding
from zhipuai_embedding import ZhipuAIEmbedding
embedding = ZhipuAIEmbedding()

#持久化路径
persist_dir = '../data_base/vector_db/chroma'

#加载数据向量库
from langchain.vectorstores.chroma import Chroma
vectordb = Chroma(
    persist_directory = persist_dir,
    embedding_function = embedding
 )
print(f'测试一下数据库，数据库的向量数目是{vectordb._collection.count()}')


/opt/conda/envs/llm-universe/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/tmp/ipykernel_16994/2526397731.py:16: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectordb = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


测试一下数据库，数据库的向量数目是1004


用as_retriever方法把数据库构造成检索器

In [3]:
retriever = vectordb.as_retriever(search_kwargs = {'k': 3})
#测试问题
question = '什么是prompt engineering?'
docs = retriever.invoke(question)
print(docs)
print(type(docs))

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(metadata={'source': '../data_base/knowledge_db/prompt_engineering/3. 迭代优化 Iterative.md'}, page_content='具体来说，首先编写初版 Prompt，然后通过多轮调整逐步改进，直到生成了满意的结果。对于更复杂的应用，可以在多个样本上进行迭代训练，评估 Prompt 的平均表现。在应用较为成熟后，才需要采用在多个样本集上评估 Prompt 性能的方式来进行细致优化。因为这需要较高的计算资源。\n\n总之，Prompt 工程师的核心是掌握 Prompt 的迭代开发和优化技巧，而非一开始就要求100%完美。通过不断调整试错，最终找到可靠适用的 Prompt 形式才是设计 Prompt 的正确方法。\n\n读者可以在 Jupyter Notebook 上，对本章给出的示例进行实践，修改 Prompt 并观察不同输出，以深入理解 Prompt 迭代优化的过程。这会对进一步开发复杂语言模型应用提供很好的实践准备。\n\n三、英文版\n\n产品说明书'), Document(metadata={'source': '../data_base/knowledge_db/prompt_engineering/1. 简介 Introduction.md'}, page_content='第一章 简介\n\n欢迎来到面向开发者的提示工程部分，本部分内容基于吴恩达老师的《Prompt Engineering for Developer》课程进行编写。《Prompt Engineering for Developer》课程是由吴恩达老师与 OpenAI 技术团队成员 Isa Fulford 老师合作授课，Isa 老师曾开发过受欢迎的 ChatGPT 检索插件，并且在教授 LLM （Large Language Model， 大语言模型）技术在产品中的应用方面做出了很大贡献。她还参与编写了教授人们使用 Prompt 的 OpenAI cookbook。我们希望通过本模块的学习，与大家分享使用提示词开发 LLM 应用的最佳实践和技巧。'), Document(metadata={'source': '../data_base/knowledge_db/prompt_engineer

## 2创建检索链
使用LCEL，langchain要求所有组成元素都要是runnable类型，使用RunnableLambda来使函数成为该类

In [4]:
from langchain_core.runnables import RunnableLambda
#定义一个合并文档的函数, "\n\n".join()将括号内的字符用指定的符号连接起来
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
#转换为Runnable类
combiner = RunnableLambda(combine_docs)

In [5]:
#创建检索链
retriever_chain = retriever | combiner
retriever_chain.invoke("西瓜书是什么？")

'前言\n“周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读\n者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推\n导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充\n具体的推导细节。”\n读到这里，大家可能会疑问为啥前面这段话加了引号，因为这只是我们最初的遐想，后来我们了解到，周\n老师之所以省去这些推导细节的真实原因是，他本尊认为“理工科数学基础扎实点的大二下学生应该对西瓜书\n中的推导细节无困难吧，要点在书里都有了，略去的细节应能脑补或做练习”。所以...... 本南瓜书只能算是我\n等数学渣渣在自学的时候记下来的笔记，希望能够帮助大家都成为一名合格的“理工科数学基础扎实点的大二\n下学生”。\n使用说明\n• 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书\n为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书；\n• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得\n\nm\nX\ni,j=1\n(W)ij\n1\np\ndidj\nFiF⊤\nj =\nm\nX\ni,j=1\n(W)ij\n1\np\ndjdi\nFjF⊤\ni\n综上所述(以上拆分的四项中前两项相等、后两项相等, 正好抵消系数1\n2 ):\n1\n2\n\uf8eb\n\uf8ed\nm\nX\ni,j=1\n(W)ij\n\r\r\r\r\r\n1\n√di\nFi −\n1\np\ndj\nFj\n\r\r\r\r\r\n2\uf8f6\n\uf8f8= tr\n\x00FF⊤\x01\n−tr\n\x10\nSFF⊤\x11\n第2 部分:\n西瓜书中式(13.21) 的第2 部分与原文献[2] 中式(4) 的第2 部分不同:\nQ(F) = 1\n2\nn\nX\ni,j=1\nWij\n\r\r\r\r\r\nFi\n√Dii\n−\nFj\np\nDjj\n\r\r\r\r\r\n2\n+ µ\nn\nX\ni=1\n∥Fi −Yi∥2 ,\n原文献中第2 部分包含了所有样本(求和变量上限为n ), 而西瓜书只包含有标记样本, 并且第

## 3创建LLM
此处使用智谱api创建一个LLM

In [6]:
from zhipuai_llm import ZhipuaiLLM
llm = ZhipuaiLLM(
    model_name = 'glm-4',
    temperature=0.1,
    api_key=api_key
)
llm.invoke("你是否能正常使用了？")

AIMessage(content='是的，我可以正常使用！😊  \n\n如果你有任何问题或需要帮助，随时告诉我，我会尽力为你提供支持。你想了解什么，或者需要我帮你做什么呢？', response_metadata={'time_in_seconds': 2.183}, id='run-9a83710c-bf48-4c6a-93a6-b627f483e9e4-0', usage_metadata={'input_tokens': 14, 'output_tokens': 37, 'total_tokens': 51})

## 4构建检索问答链

RunnablePassthrough 相当于一种占位符，什么也不做的让上行输入流入；RunnableParallel就是让上行输入并行流入内部字典的键值中，然后返回字典

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_core.output_parsers import StrOutputParser

template = """请参考以下上下文来回答最后的问题，如果你不知道就说你不知道。尽量使答案简明扼要。
{context}
问题：{input}
"""
#提示词转换为Runnable类
prompt = PromptTemplate(template=template)

#问答链
qa_chain = (
    #并行输入
    RunnableParallel({"context": retriever_chain,"input":RunnablePassthrough()})
    | prompt
    | llm
    | StrOutputParser()
)

In [8]:
#测试一下
question_test = "讲讲提示词工程"
result = qa_chain.invoke(question_test)
print(result)

提示词工程是设计和优化输入文本（提示词）以引导AI模型生成更准确、有用输出的过程。关键包括明确任务、提供上下文、控制输出格式，并通过迭代测试改进效果。


## 5向检索链中添加聊天记录
使用langchain中的`ChatPromptTemplate`接收聊天消息记录并传给LLM
 

In [9]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = (
    "你是一个问答任务的助手。 "
    "请使用检索到的上下文片段回答这个问题。 "
    "如果你不知道答案就说不知道。 "
    "请使用简洁的话语回答用户。"    
    "\n\n"
    "{context}"
 )

qa_prompt = ChatPromptTemplate(
    [
        ("system",system_prompt),
        ("placeholder","{chat_history}"),
        ("human","{input}")
    ]
)
#传入历史记录

messages = qa_prompt.invoke(
    {
        "input":"你可以介绍一下西瓜书吗？",
        "context":"",
        "chat_history":[
        ("human","西瓜书是什么？"),
        ("ai","西瓜书是周志华写的一本书。")
        ]
    }
)
for message in messages.messages:
    print(message.content)

你是一个问答任务的助手。 请使用检索到的上下文片段回答这个问题。 如果你不知道答案就说不知道。 请使用简洁的话语回答用户。


西瓜书是什么？
西瓜书是周志华写的一本书。
你可以介绍一下西瓜书吗？


## 6带有信息压缩的检索链
在多轮对话中。用户经常采取简短的（语义不全的）询问，这种问题在查询向量数据库时很难检索到相关信息。因此使用`信息压缩`来使llm根据历史记录来补全用户的问题

In [12]:
from langchain_core.runnables import RunnableBranch
condense_question_system_prompt = (
    "请根据用户的聊天记录完善用户最新的问题，"
    "如果用户最新的问题不需要完善则返回用户的问题。"
)

condense_question_prompt = ChatPromptTemplate([
    ("system",condense_question_system_prompt),
    ("placeholder","{chat_history}"),
    ("human","{input}")
]
)

#构造检索链
retrieve_docs = RunnableBranch(
    #RunnalbleBranch: (（判断，默认行为），默认流向，)
    #分支1：若聊天记录中没有历史记录，则直接将用户输入传递给检索链
    (lambda x: not x.get("chat_history",False),(lambda x: x["input"])|retriever,),
    #分支2：若聊天记录中有历史记录，则送往llm补全再传给检索链
    condense_question_prompt|llm|StrOutputParser()|retriever,
)

In [ ]:
# 重新定义 combine_docs
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs["context"]) # 将 docs 改为 docs["context"]
# 定义问答链
qa_chain = (
    RunnablePassthrough.assign(context=combine_docs) # 使用 combine_docs 函数整合 qa_prompt 中的 context
    | qa_prompt # 问答模板
    | llm
    | StrOutputParser() # 规定输出的格式为 str
) 
# 定义带有历史记录的问答链
qa_history_chain = RunnablePassthrough.assign(
    context = (lambda x: x) | retrieve_docs # 将查询结果存为 content
    ).assign(answer=qa_chain) # 将最终结果存为 answer

# 不带聊天记录
qa_history_chain.invoke({
    "input": "西瓜书是什么？",
    "chat_history": []
})

{'input': '西瓜书是什么？',
 'chat_history': [],
 'context': [Document(metadata={'author': '', 'creationDate': "D:20230303170709-00'00'", 'creator': 'LaTeX with hyperref', 'file_path': '../data_base/knowledge_db/pumkin_book/pumpkin_book.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': '', 'page': 1, 'producer': 'xdvipdfmx (20200315)', 'source': '../data_base/knowledge_db/pumkin_book/pumpkin_book.pdf', 'subject': '', 'title': '', 'total_pages': 196, 'trapped': ''}, page_content='前言\n“周志华老师的《机器学习》（西瓜书）是机器学习领域的经典入门教材之一，周老师为了使尽可能多的读\n者通过西瓜书对机器学习有所了解, 所以在书中对部分公式的推导细节没有详述，但是这对那些想深究公式推\n导细节的读者来说可能“不太友好”，本书旨在对西瓜书里比较难理解的公式加以解析，以及对部分公式补充\n具体的推导细节。”\n读到这里，大家可能会疑问为啥前面这段话加了引号，因为这只是我们最初的遐想，后来我们了解到，周\n老师之所以省去这些推导细节的真实原因是，他本尊认为“理工科数学基础扎实点的大二下学生应该对西瓜书\n中的推导细节无困难吧，要点在书里都有了，略去的细节应能脑补或做练习”。所以...... 本南瓜书只能算是我\n等数学渣渣在自学的时候记下来的笔记，希望能够帮助大家都成为一名合格的“理工科数学基础扎实点的大二\n下学生”。\n使用说明\n• 南瓜书的所有内容都是以西瓜书的内容为前置知识进行表述的，所以南瓜书的最佳使用方法是以西瓜书\n为主线，遇到自己推导不出来或者看不懂的公式时再来查阅南瓜书；\n• 对于初学机器学习的小白，西瓜书第1 章和第2 章的公式强烈不建议深究，简单过一下即可，等你学得'),
  Do